In [ ]:

import gondola as gon
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

run_id = 251
paddles = gon.db.TofPaddle.all()

#%run prelude.rc

import enum
import importlib.util
import sys
from pathlib import Path


import polars as pl
import numpy as np
import scipy.integrate as integrate
import HErmes as he
import HErmes.fitting as fit
import scipy.stats as st
import matplotlib

from scipy.spatial.transform import Rotation as rot
from datetime import datetime, UTC, timezone
from glob import glob

#pybindings
from pathlib import Path
import dashi as d
d.visual()
import tqdm


import matplotlib.pyplot as plt
import charmingbeauty as cb
lo = cb.layout
cb.visual.set_style_present()


import re
!export DJANGO_ALLOW_ASYNC_UNSAFE=1
import os
from matplotlib import font_manager
from matplotlib import rcParams


os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = '1'
plt.rcParams.update({'text.usetex' : False})


from matplotlib import font_manager


rcParams['font.family'] = 'sans-serif'
rcParams['font.sans-serif'] = ['Open Sans']


import gondola as gon
import time

files = gon.io.grace_get_telemetry_binaries(
    1766035400,
    1766135400,
    #1765835400,
    #1767979800, #end time
    '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
)


lpt = None
toml_find = False

Num  = 200



sigB1_list = []
sigB2_list = []
sigReal1_list = []
sigReal2_list = []



time_a_arr = []
time_b_arr = []

filecount = 0
#fileTot = len(files)

for f in files:  # in files from flight
    filecount = filecount + 1
    reader = gon.io.TelemetryPacketReader(f)

    if filecount % 300 == 0:
        print(f"processed {filecount} files!")
    
    for pack in reader:
        if (
            pack.packet_type == gon.packets.TelemetryPacketType.BoringEvent
            or pack.packet_type == gon.packets.TelemetryPacketType.InterestingEvent
        ):
            toml_find = False
            ev = gon.events.TelemetryEvent.from_telemetrypacket(pack)
            for hit in ev.tof.hits:
                #print(hit)
                
                mu1 = hit.baseline_a
                sigB1 = hit.baseline_a_rms

                mu2 = hit.baseline_b
                sigB2 = hit.baseline_b_rms
                
                sigReal1_sq = ((Num - mu1**2) / Num) * sigB1**2 - mu1**2
                sigReal2_sq = ((Num - mu2**2) / Num) * sigB2**2 - mu2**2
                
                if sigReal1_sq >= 0:
                    sigReal1 = np.sqrt(sigReal1_sq)
                else:
                    sigReal1 = np.nan
                
                if sigReal2_sq >= 0:
                    sigReal2 = np.sqrt(sigReal2_sq)
                else:
                    sigReal2 = np.nan
                
                #sigB1_list.append(sigB1)
                #sigB2_list.append(sigB2)
                #sigReal1_list.append(sigReal1)
                #sigReal2_list.append(sigReal2)
                time_a_arr.append(hit.time_a)
                time_b_arr.append(hit.time_b)

                
                

In [ ]:
%matplotlib inline
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]  # always available


time_a_vals = np.asarray(time_a_arr)
time_b_vals = np.asarray(time_b_arr)

# -------------------------------------------------
# Histograms: packet baseline RMS vs derived sigma
# -------------------------------------------------

plt.figure(figsize=(8, 5))
plt.hist(time_a_vals, bins=80, histtype="step", label="hit.time_a_vals")
plt.hist(time_b_vals, bins=80, histtype="step", label="hit.time_b_vals")
plt.xlabel("time")
plt.ylabel("Counts")
plt.title("Side A and B timing")
plt.legend()
#plt.xlim(0,7)
plt.yscale("log")
plt.tight_layout()
plt.show()

